In [ ]:
# ============================================================
# RECUPERAÇÃO RAIS_PROFESSORES — 2022 — V2
# CINCO FAMÍLIAS CBO + COMPARAÇÃO COM A BASE FINAL V2
# ============================================================
#
# OBJETIVOS
# ------------------------------------------------------------
# 1. Reprocessar SOMENTE 2022.
# 2. Filtrar as cinco famílias CBO usadas na especificação final:
#       2312, 2313, 2321, 3312 e 3321.
# 3. Preservar TODAS as colunas originais da RAIS.
# 4. Acrescentar campos padronizados para a futura base V3.
# 5. NÃO criar nem recalcular Y_doenca.
# 6. Comparar a população recuperada de 2022 com a base
#    RAIS_BASE_MODELO_FINAL_V2:
#       - total de vínculos;
#       - distribuição por UF;
#       - distribuição por Família CBO;
#       - distribuição por UF x Família CBO;
#       - apenas como referência, prevalência de Y_doenca na V2.
# 7. Gravar os novos Parquets em uma pasta separada, sem
#    sobrescrever os arquivos anteriores de 2022.
#
# IMPORTANTE
# ------------------------------------------------------------
# - Os campos de afastamento são preservados porque fazem parte
#   do arquivo original, mas NÃO são usados como preditores aqui.
# - Este script NÃO altera a base final V2.
# - Este script NÃO cria ainda a base V3.
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive
from ftplib import FTP
from urllib.parse import quote

from collections import Counter

import gc
import glob
import os
import re
import shutil
import subprocess
import unicodedata

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

from IPython.display import display


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False
)


# ============================================================
# 3. CONFIGURAÇÕES
# ============================================================

ANO = 2022

PASTA_TCC = (
    "/content/drive/MyDrive/TCC_2"
)

PASTA_DADOS = os.path.join(
    PASTA_TCC,
    "dados"
)

# Onde procurar compactados já baixados.
PASTA_COMPACTADOS = os.path.join(
    PASTA_DADOS,
    "RAIS_OUTROS_ESTADOS"
)

# NOVA pasta candidata para os Parquets de 2022.
# Não sobrescreve RAIS_PROFESSORES/2022.
PASTA_SAIDA_PARQUETS = os.path.join(
    PASTA_DADOS,
    "RAIS_PROFESSORES_V3_CANDIDATA",
    str(ANO)
)

# Base final atual usada como referência.
PASTA_BASE_V2 = os.path.join(
    PASTA_DADOS,
    "RAIS_BASE_MODELO_FINAL_V2"
)

PASTA_RESULTADOS = os.path.join(
    PASTA_TCC,
    "resultados",
    "RECUPERACAO_RAIS_PROFESSORES_2022_V2"
)

PASTA_TEMP = (
    "/content/RAIS_TEMP_2022_V2"
)


# FTP oficial como fallback.
HOST = "ftp.mtps.gov.br"
BASE_FTP = "/pdet/microdados/RAIS"


# Leitura em blocos.
CHUNKSIZE = 300_000


# Se True, refaz os seis Parquets candidatos mesmo que já existam.
SOBRESCREVER_2022 = True


GRUPOS = {
    "NORTE":
        "RAIS_VINC_PUB_NORTE",

    "NORDESTE":
        "RAIS_VINC_PUB_NORDESTE",

    "CENTRO_OESTE":
        "RAIS_VINC_PUB_CENTRO_OESTE",

    "MG_ES_RJ":
        "RAIS_VINC_PUB_MG_ES_RJ",

    "SP":
        "RAIS_VINC_PUB_SP",

    "SUL":
        "RAIS_VINC_PUB_SUL",
}


# Cinco famílias presentes na especificação final.
FAMILIAS_CBO = {

    "2312":
        "Professores de nível superior do ensino fundamental - anos iniciais",

    "2313":
        "Professores de nível superior do ensino fundamental - anos finais",

    "2321":
        "Professores do ensino médio",

    "3312":
        "Professores de nível médio no ensino fundamental",

    "3321":
        "Professores leigos no ensino fundamental",
}


MAPA_UF = {

    "11": "RO",
    "12": "AC",
    "13": "AM",
    "14": "RR",
    "15": "PA",
    "16": "AP",
    "17": "TO",

    "21": "MA",
    "22": "PI",
    "23": "CE",
    "24": "RN",
    "25": "PB",
    "26": "PE",
    "27": "AL",
    "28": "SE",
    "29": "BA",

    "31": "MG",
    "32": "ES",
    "33": "RJ",
    "35": "SP",

    "41": "PR",
    "42": "SC",
    "43": "RS",

    "50": "MS",
    "51": "MT",
    "52": "GO",
    "53": "DF",
}


UFS_ESPERADAS = {

    "NORTE": {
        "AC", "AP", "AM", "PA",
        "RO", "RR", "TO"
    },

    "NORDESTE": {
        "AL", "BA", "CE", "MA",
        "PB", "PE", "PI", "RN", "SE"
    },

    "CENTRO_OESTE": {
        "DF", "GO", "MT", "MS"
    },

    "MG_ES_RJ": {
        "MG", "ES", "RJ"
    },

    "SP": {
        "SP"
    },

    "SUL": {
        "PR", "SC", "RS"
    },
}


for pasta in [
    PASTA_COMPACTADOS,
    PASTA_SAIDA_PARQUETS,
    PASTA_RESULTADOS,
    PASTA_TEMP,
]:
    os.makedirs(
        pasta,
        exist_ok=True
    )


# ============================================================
# 4. INSTALAR 7ZIP, SE NECESSÁRIO
# ============================================================

if shutil.which("7z") is None:

    subprocess.run(
        [
            "apt-get",
            "update"
        ],
        stdout=subprocess.DEVNULL,
        check=True
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "p7zip-full"
        ],
        stdout=subprocess.DEVNULL,
        check=True
    )


# ============================================================
# 5. UTILITÁRIOS
# ============================================================

def normalizar_texto(
    texto
):

    texto = unicodedata.normalize(
        "NFKD",
        str(texto)
    )

    texto = "".join(
        c
        for c in texto
        if not unicodedata.combining(c)
    )

    texto = texto.upper()

    texto = re.sub(
        r"[^A-Z0-9]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    ).strip()

    return texto


def extrair_codigo_6(
    serie
):

    return (
        serie
        .astype("string")
        .str.extract(
            r"(\d{6})",
            expand=False
        )
    )


def extrair_familia_4(
    serie
):

    codigo = (
        serie
        .astype("string")
        .str.extract(
            r"(\d{4,6})",
            expand=False
        )
    )

    return (
        codigo
        .str[:4]
    )


def detectar_coluna(
    colunas,
    candidatos_exatos=None,
    contem_todos=None,
    obrigatoria=False,
    descricao=""
):

    candidatos_exatos = (
        candidatos_exatos
        or []
    )

    contem_todos = (
        contem_todos
        or []
    )

    mapa = {
        normalizar_texto(col):
            col
        for col in colunas
    }

    # Primeiro: nomes exatos normalizados.
    for candidato in candidatos_exatos:

        chave = normalizar_texto(
            candidato
        )

        if chave in mapa:

            return mapa[
                chave
            ]

    # Depois: todos os termos precisam aparecer.
    for col in colunas:

        nome = normalizar_texto(
            col
        )

        for termos in contem_todos:

            if all(
                termo in nome
                for termo in termos
            ):

                return col

    if obrigatoria:

        raise RuntimeError(
            f"Não encontrei a coluna obrigatória: {descricao}"
        )

    return None


# ============================================================
# 6. PROCURAR ARQUIVO BRUTO LOCAL
# ============================================================

def localizar_bruto_local(
    nome_base
):

    candidatos = glob.glob(
        os.path.join(
            PASTA_DADOS,
            "**",
            f"{nome_base}*"
        ),
        recursive=True,
    )

    validos = []

    for caminho in candidatos:

        if not os.path.isfile(
            caminho
        ):
            continue

        nome = (
            os.path.basename(
                caminho
            )
            .lower()
        )

        # Não usar compactados nem Parquets como "bruto texto".
        if nome.endswith(
            (
                ".7z",
                ".zip",
                ".parquet"
            )
        ):
            continue

        partes = (
            os.path.normpath(
                caminho
            )
            .split(
                os.sep
            )
        )

        if str(
            ANO
        ) not in partes:

            continue

        validos.append(
            caminho
        )

    if not validos:

        return None

    return max(
        validos,
        key=os.path.getsize
    )


# ============================================================
# 7. FTP COMO FALLBACK
# ============================================================

def listar_ftp(
    ano
):

    erros = []

    for encoding in [
        "utf-8",
        "cp1252",
        "latin1"
    ]:

        ftp = None

        try:

            ftp = FTP(
                HOST,
                timeout=180,
                encoding=encoding
            )

            ftp.login()

            ftp.cwd(
                f"{BASE_FTP}/{ano}"
            )

            arquivos = (
                ftp.nlst()
            )

            ftp.quit()

            return arquivos

        except Exception as erro:

            erros.append(
                f"{encoding}: {erro}"
            )

            try:

                if ftp is not None:
                    ftp.close()

            except Exception:
                pass

    raise RuntimeError(
        "Não foi possível listar o FTP:\n"
        +
        "\n".join(
            erros
        )
    )


def localizar_arquivo_ftp(
    arquivos,
    nome_base
):

    candidatos = [

        arquivo
        for arquivo in arquivos

        if (
            nome_base.upper()
            in os.path.basename(
                arquivo
            ).upper()
        )

        and (
            os.path.basename(
                arquivo
            )
            .upper()
            .endswith(
                ".7Z"
            )
        )
    ]

    if len(
        candidatos
    ) != 1:

        raise RuntimeError(
            f"Esperava exatamente um .7z para {nome_base}. "
            f"Candidatos: {candidatos}"
        )

    return candidatos[
        0
    ]


def obter_compactado(
    grupo,
    nome_base,
    arquivos_ftp
):

    locais = glob.glob(
        os.path.join(
            PASTA_DADOS,
            "**",
            f"*{nome_base}*.7z"
        ),
        recursive=True,
    )

    locais = [
        x
        for x in locais
        if str(
            ANO
        )
        in os.path.normpath(
            x
        ).split(
            os.sep
        )
    ]

    if locais:

        return max(
            locais,
            key=os.path.getsize
        )

    remoto = localizar_arquivo_ftp(
        arquivos_ftp,
        nome_base
    )

    nome_remoto = (
        os.path.basename(
            remoto
        )
    )

    pasta_grupo = os.path.join(
        PASTA_COMPACTADOS,
        grupo,
        str(
            ANO
        )
    )

    os.makedirs(
        pasta_grupo,
        exist_ok=True
    )

    destino = os.path.join(
        pasta_grupo,
        nome_remoto
    )

    url = (
        f"ftp://{HOST}"
        f"{BASE_FTP}/{ANO}/"
        f"{quote(nome_remoto)}"
    )

    print(
        "Baixando:"
    )

    print(
        url
    )

    subprocess.run(
        [
            "wget",
            "--continue",
            "--progress=bar:force",
            "-O",
            destino,
            url
        ],
        check=True,
    )

    return destino


def extrair_compactado(
    arquivo_7z,
    grupo
):

    destino = os.path.join(
        PASTA_TEMP,
        grupo
    )

    if os.path.exists(
        destino
    ):

        shutil.rmtree(
            destino
        )

    os.makedirs(
        destino,
        exist_ok=True
    )

    subprocess.run(
        [
            "7z",
            "x",
            arquivo_7z,
            f"-o{destino}",
            "-y"
        ],
        check=True
    )

    extraidos = []

    for raiz, _, nomes in os.walk(
        destino
    ):

        for nome in nomes:

            caminho = os.path.join(
                raiz,
                nome
            )

            if os.path.isfile(
                caminho
            ):

                extraidos.append(
                    caminho
                )

    if len(
        extraidos
    ) != 1:

        raise RuntimeError(
            f"O .7z de {grupo} gerou "
            f"{len(extraidos)} arquivos. "
            f"Revisar: {extraidos}"
        )

    return extraidos[
        0
    ]


# ============================================================
# 8. IDENTIFICAR O LAYOUT DO ARQUIVO BRUTO DE 2022
# ============================================================

def identificar_layout(
    caminho
):

    for encoding in [
        "utf-8",
        "cp1252",
        "latin1"
    ]:

        for sep in [
            ";",
            ",",
            "\t",
            "|"
        ]:

            try:

                cab = pd.read_csv(
                    caminho,
                    sep=sep,
                    encoding=encoding,
                    nrows=0
                )

                colunas = list(
                    cab.columns
                )

                if len(
                    colunas
                ) < 20:

                    continue

                cbo = detectar_coluna(
                    colunas,
                    candidatos_exatos=[
                        "CBO Ocupação 2002",
                        "CBO 2002 Ocupação - Código",
                    ],
                    contem_todos=[
                        [
                            "CBO",
                            "2002",
                            "OCUP"
                        ]
                    ],
                    obrigatoria=True,
                    descricao="CBO 2002"
                )

                municipio_estab = detectar_coluna(
                    colunas,
                    candidatos_exatos=[
                        "Município",
                        "Município - Código",
                        "Municipio",
                        "Municipio - Codigo",
                    ],
                    contem_todos=[],
                    obrigatoria=True,
                    descricao="Município do estabelecimento"
                )

                municipio_trab = detectar_coluna(
                    colunas,
                    candidatos_exatos=[
                        "Mun Trab",
                        "Município Trab",
                        "Município Trabalhador",
                        "Municipio Trab",
                    ],
                    contem_todos=[
                        [
                            "MUN",
                            "TRAB"
                        ]
                    ],
                    obrigatoria=False,
                    descricao="Município do trabalhador"
                )

                return {
                    "encoding":
                        encoding,

                    "sep":
                        sep,

                    "cbo":
                        cbo,

                    "municipio_estab":
                        municipio_estab,

                    "municipio_trab":
                        municipio_trab,

                    "colunas":
                        colunas,
                }

            except RuntimeError:
                raise

            except Exception:
                continue

    raise RuntimeError(
        "Não foi possível identificar o layout de "
        f"{caminho}"
    )


# ============================================================
# 9. PROCESSAR UM DOS SEIS GRUPOS
# ============================================================

def processar_grupo(
    caminho_bruto,
    grupo,
    saida_parquet
):

    layout = identificar_layout(
        caminho_bruto
    )

    print(
        "\nLayout detectado:"
    )

    print(
        "  encoding:",
        layout[
            "encoding"
        ]
    )

    print(
        "  separador:",
        repr(
            layout[
                "sep"
            ]
        )
    )

    print(
        "  CBO:",
        layout[
            "cbo"
        ]
    )

    print(
        "  Município estabelecimento:",
        layout[
            "municipio_estab"
        ]
    )

    print(
        "  Município trabalhador:",
        layout[
            "municipio_trab"
        ]
    )

    print(
        "  colunas brutas:",
        len(
            layout[
                "colunas"
            ]
        )
    )


    # Salvar schema bruto.
    pd.DataFrame(
        {
            "Posicao":
                range(
                    len(
                        layout[
                            "colunas"
                        ]
                    )
                ),

            "Coluna":
                layout[
                    "colunas"
                ],
        }
    ).to_csv(
        os.path.join(
            PASTA_RESULTADOS,
            f"schema_bruto_2022_{grupo}.csv"
        ),
        index=False,
        encoding="utf-8-sig",
    )


    auditoria = {

        "Ano":
            ANO,

        "Grupo":
            grupo,

        "Linhas_brutas":
            0,

        "Professores_5_familias":
            0,

        "UF_estab_nula":
            0,

        "UF_trab_nula":
            0,

        "Municipio_estab_nulo":
            0,
    }


    contagem_familia = Counter()

    contagem_uf = Counter()

    contagem_uf_familia = Counter()


    tmp = (
        saida_parquet
        +
        ".tmp"
    )

    if os.path.exists(
        tmp
    ):

        os.remove(
            tmp
        )


    writer = None

    total_prof = 0


    try:

        leitor = pd.read_csv(
            caminho_bruto,
            sep=layout[
                "sep"
            ],
            encoding=layout[
                "encoding"
            ],
            dtype=str,
            chunksize=CHUNKSIZE,
            low_memory=False,
            on_bad_lines="error",
        )


        for i, chunk in enumerate(
            leitor,
            start=1
        ):

            auditoria[
                "Linhas_brutas"
            ] += len(
                chunk
            )


            # ------------------------------------------------
            # CBO e Família CBO
            # ------------------------------------------------

            cbo = (

                chunk[
                    layout[
                        "cbo"
                    ]
                ]

                .astype(
                    "string"
                )

                .str.extract(
                    r"(\d{6})",
                    expand=False
                )
            )


            familia = (
                cbo
                .str[:4]
            )


            mascara = (
                familia
                .isin(
                    FAMILIAS_CBO.keys()
                )
            )


            prof = (
                chunk
                .loc[
                    mascara
                ]
                .copy()
            )


            if prof.empty:

                print(
                    f"Chunk {i}: "
                    f"{len(chunk):,} linhas | "
                    "0 professores"
                )

                del (
                    chunk,
                    cbo,
                    familia,
                    mascara
                )

                gc.collect()

                continue


            # ------------------------------------------------
            # Campos padronizados
            # ------------------------------------------------

            prof[
                "CBO_padronizada"
            ] = (
                cbo
                .loc[
                    mascara
                ]
                .values
            )


            prof[
                "Familia_CBO"
            ] = (
                familia
                .loc[
                    mascara
                ]
                .values
            )


            prof[
                "Descricao_Familia_CBO"
            ] = (

                prof[
                    "Familia_CBO"
                ]

                .map(
                    FAMILIAS_CBO
                )
            )


            prof[
                "Ano"
            ] = str(
                ANO
            )


            prof[
                "Grupo_Origem"
            ] = grupo


            # ------------------------------------------------
            # Município e UF do estabelecimento
            # ------------------------------------------------

            mun_estab = extrair_codigo_6(
                prof[
                    layout[
                        "municipio_estab"
                    ]
                ]
            )


            prof[
                "Municipio_estabelecimento_codigo"
            ] = (
                mun_estab
            )


            prof[
                "UF_estabelecimento"
            ] = (

                mun_estab
                .str[:2]
                .map(
                    MAPA_UF
                )
            )


            # UF canônica para a V3:
            # localização do estabelecimento.
            prof[
                "UF"
            ] = (
                prof[
                    "UF_estabelecimento"
                ]
            )


            # ------------------------------------------------
            # Município do trabalhador
            # Preservado separadamente.
            # ------------------------------------------------

            if (
                layout[
                    "municipio_trab"
                ]
                is not None
            ):

                mun_trab = extrair_codigo_6(
                    prof[
                        layout[
                            "municipio_trab"
                        ]
                    ]
                )


                prof[
                    "Municipio_trabalhador_codigo"
                ] = (
                    mun_trab
                )


                prof[
                    "UF_trabalhador"
                ] = (

                    mun_trab
                    .str[:2]
                    .map(
                        MAPA_UF
                    )
                )

            else:

                prof[
                    "Municipio_trabalhador_codigo"
                ] = pd.NA

                prof[
                    "UF_trabalhador"
                ] = pd.NA


            # ------------------------------------------------
            # Auditoria
            # ------------------------------------------------

            auditoria[
                "Professores_5_familias"
            ] += len(
                prof
            )


            auditoria[
                "UF_estab_nula"
            ] += int(
                prof[
                    "UF_estabelecimento"
                ]
                .isna()
                .sum()
            )


            auditoria[
                "UF_trab_nula"
            ] += int(
                prof[
                    "UF_trabalhador"
                ]
                .isna()
                .sum()
            )


            auditoria[
                "Municipio_estab_nulo"
            ] += int(
                prof[
                    "Municipio_estabelecimento_codigo"
                ]
                .isna()
                .sum()
            )


            # ------------------------------------------------
            # Contagens para comparação com a V2
            # ------------------------------------------------

            fam_counts = (

                prof[
                    "Familia_CBO"
                ]

                .astype(
                    "string"
                )

                .value_counts(
                    dropna=False
                )
            )


            for fam, qtd in (
                fam_counts.items()
            ):

                contagem_familia[
                    str(
                        fam
                    )
                ] += int(
                    qtd
                )


            uf_counts = (

                prof[
                    "UF"
                ]

                .astype(
                    "string"
                )

                .value_counts(
                    dropna=False
                )
            )


            for uf, qtd in (
                uf_counts.items()
            ):

                contagem_uf[
                    str(
                        uf
                    )
                ] += int(
                    qtd
                )


            uf_fam_counts = (

                prof

                .groupby(
                    [
                        "UF",
                        "Familia_CBO"
                    ],
                    dropna=False
                )

                .size()
            )


            for (
                uf,
                fam
            ), qtd in (
                uf_fam_counts.items()
            ):

                contagem_uf_familia[
                    (
                        str(
                            uf
                        ),
                        str(
                            fam
                        )
                    )
                ] += int(
                    qtd
                )


            # ------------------------------------------------
            # Schema homogêneo entre chunks
            # ------------------------------------------------

            for coluna in prof.columns:

                prof[
                    coluna
                ] = (
                    prof[
                        coluna
                    ]
                    .astype(
                        "string"
                    )
                )


            tabela = (
                pa.Table
                .from_pandas(
                    prof,
                    preserve_index=False
                )
            )


            if writer is None:

                writer = pq.ParquetWriter(
                    tmp,
                    tabela.schema,
                    compression="snappy"
                )


            writer.write_table(
                tabela
            )


            total_prof += len(
                prof
            )


            print(
                f"Chunk {i}: "
                f"{len(chunk):,} linhas | "
                f"{len(prof):,} professores | "
                f"acumulado={total_prof:,}"
            )


            del (
                chunk,
                prof,
                tabela,
                cbo,
                familia,
                mascara,
                mun_estab
            )

            if (
                "mun_trab"
                in locals()
            ):

                del mun_trab

            gc.collect()


    finally:

        if writer is not None:

            writer.close()


    if not os.path.exists(
        tmp
    ):

        raise RuntimeError(
            "Nenhum registro foi gravado."
        )


    pf = pq.ParquetFile(
        tmp
    )


    if (
        pf.metadata.num_rows
        !=
        total_prof
    ):

        raise RuntimeError(
            f"Divergência: "
            f"{total_prof:,} filtrados vs "
            f"{pf.metadata.num_rows:,} gravados."
        )


    obrigatorias = {

        "CBO_padronizada",
        "Familia_CBO",
        "Descricao_Familia_CBO",

        "Ano",
        "Grupo_Origem",

        "Municipio_estabelecimento_codigo",
        "UF_estabelecimento",
        "UF",

        "Municipio_trabalhador_codigo",
        "UF_trabalhador",
    }


    faltantes = (

        obrigatorias
        -
        set(
            pf.schema_arrow.names
        )
    )


    if faltantes:

        raise RuntimeError(
            "Campos padronizados ausentes: "
            f"{sorted(faltantes)}"
        )


    # --------------------------------------------------------
    # Publicação atômica
    # --------------------------------------------------------

    if os.path.exists(
        saida_parquet
    ):

        os.remove(
            saida_parquet
        )


    os.replace(
        tmp,
        saida_parquet
    )


    auditoria[
        "Qtd_colunas_brutas"
    ] = len(
        layout[
            "colunas"
        ]
    )


    auditoria[
        "Qtd_colunas_parquet"
    ] = len(
        pq.ParquetFile(
            saida_parquet
        )
        .schema_arrow.names
    )


    auditoria[
        "Arquivo_saida"
    ] = (
        saida_parquet
    )


    # --------------------------------------------------------
    # DataFrames das distribuições
    # --------------------------------------------------------

    df_familia = pd.DataFrame(
        [
            {
                "Ano":
                    ANO,

                "Grupo":
                    grupo,

                "Familia_CBO":
                    fam,

                "Descricao":
                    FAMILIAS_CBO.get(
                        fam,
                        "Outra"
                    ),

                "N":
                    qtd,
            }

            for fam, qtd
            in contagem_familia.items()
        ]
    )


    df_uf = pd.DataFrame(
        [
            {
                "Ano":
                    ANO,

                "Grupo":
                    grupo,

                "UF":
                    uf,

                "N":
                    qtd,
            }

            for uf, qtd
            in contagem_uf.items()
        ]
    )


    df_uf_familia = pd.DataFrame(
        [
            {
                "Ano":
                    ANO,

                "Grupo":
                    grupo,

                "UF":
                    uf,

                "Familia_CBO":
                    fam,

                "Descricao":
                    FAMILIAS_CBO.get(
                        fam,
                        "Outra"
                    ),

                "N":
                    qtd,
            }

            for (
                uf,
                fam
            ), qtd
            in contagem_uf_familia.items()
        ]
    )


    return (
        auditoria,
        df_familia,
        df_uf,
        df_uf_familia,
    )


# ============================================================
# 10. VALIDAR PARQUET CANDIDATO EXISTENTE
# ============================================================

def parquet_valido(
    caminho
):

    if not os.path.exists(
        caminho
    ):

        return False


    try:

        pf = pq.ParquetFile(
            caminho
        )


        obrigatorias = {
            "CBO_padronizada",
            "Familia_CBO",
            "Ano",
            "Grupo_Origem",
            "UF",
            "Municipio_estabelecimento_codigo",
        }


        return (

            pf.metadata.num_rows
            >
            0

            and

            obrigatorias.issubset(
                set(
                    pf.schema_arrow.names
                )
            )
        )


    except Exception:

        return False


# ============================================================
# 11. PROCESSAR OS SEIS GRUPOS DE 2022
# ============================================================

resumo = []

familias = []

ufs = []

ufs_familias = []

arquivos_ftp = None


for pos, (
    grupo,
    nome_base
) in enumerate(
    GRUPOS.items(),
    start=1
):

    print(
        "\n"
        +
        "#" * 90
    )

    print(
        f"[{pos}/6] "
        f"2022 — {grupo}"
    )

    print(
        "#" * 90
    )


    saida = os.path.join(
        PASTA_SAIDA_PARQUETS,
        f"RAIS_PROFESSORES_{ANO}_{grupo}.parquet"
    )


    # --------------------------------------------------------
    # Se já existe e não queremos sobrescrever, apenas auditar.
    # --------------------------------------------------------

    if (
        parquet_valido(
            saida
        )
        and
        not SOBRESCREVER_2022
    ):

        pf = pq.ParquetFile(
            saida
        )

        print(
            f"Já existe e é válido: "
            f"{pf.metadata.num_rows:,} registros"
        )

        resumo.append(
            {
                "Ano":
                    ANO,

                "Grupo":
                    grupo,

                "Professores_5_familias":
                    int(
                        pf.metadata.num_rows
                    ),

                "Status":
                    "JÁ EXISTIA",

                "Arquivo_saida":
                    saida,
            }
        )

        continue


    # --------------------------------------------------------
    # Procurar bruto local
    # --------------------------------------------------------

    bruto = localizar_bruto_local(
        nome_base
    )

    temp_extraido = None


    if bruto is not None:

        print(
            "Bruto encontrado no Drive:"
        )

        print(
            bruto
        )


    else:

        # ----------------------------------------------------
        # Se não existe descompactado, procurar/baixar .7z
        # ----------------------------------------------------

        if arquivos_ftp is None:

            print(
                "Bruto descompactado não encontrado. "
                "Consultando FTP de 2022..."
            )

            arquivos_ftp = (
                listar_ftp(
                    ANO
                )
            )


        compactado = obter_compactado(
            grupo,
            nome_base,
            arquivos_ftp
        )


        bruto = extrair_compactado(
            compactado,
            grupo
        )


        temp_extraido = (
            os.path.dirname(
                bruto
            )
        )


        print(
            "Extraído temporariamente:"
        )

        print(
            bruto
        )


    try:

        (
            aud,
            df_familia_grupo,
            df_uf_grupo,
            df_uf_familia_grupo,
        ) = processar_grupo(
            bruto,
            grupo,
            saida
        )


        aud[
            "Status"
        ] = "OK"


        resumo.append(
            aud
        )


        familias.append(
            df_familia_grupo
        )


        ufs.append(
            df_uf_grupo
        )


        ufs_familias.append(
            df_uf_familia_grupo
        )


    except Exception as erro:

        print(
            "ERRO:",
            erro
        )


        tmp = (
            saida
            +
            ".tmp"
        )


        if os.path.exists(
            tmp
        ):

            os.remove(
                tmp
            )


        resumo.append(
            {
                "Ano":
                    ANO,

                "Grupo":
                    grupo,

                "Professores_5_familias":
                    None,

                "Status":
                    f"ERRO: {erro}",

                "Arquivo_saida":
                    saida,
            }
        )


    finally:

        if (
            temp_extraido is not None
            and
            temp_extraido.startswith(
                PASTA_TEMP
            )
        ):

            shutil.rmtree(
                temp_extraido,
                ignore_errors=True
            )


        gc.collect()


# ============================================================
# 12. CONSOLIDAR RESULTADOS DA NOVA RECUPERAÇÃO
# ============================================================

df_resumo = pd.DataFrame(
    resumo
)


if familias:

    df_familia_novo = (

        pd.concat(
            familias,
            ignore_index=True
        )

        .groupby(
            [
                "Ano",
                "Familia_CBO",
                "Descricao"
            ],
            as_index=False
        )[
            "N"
        ]

        .sum()
    )

else:

    df_familia_novo = pd.DataFrame(
        columns=[
            "Ano",
            "Familia_CBO",
            "Descricao",
            "N"
        ]
    )


if ufs:

    df_uf_novo = (

        pd.concat(
            ufs,
            ignore_index=True
        )

        .groupby(
            [
                "Ano",
                "UF"
            ],
            as_index=False
        )[
            "N"
        ]

        .sum()
    )

else:

    df_uf_novo = pd.DataFrame(
        columns=[
            "Ano",
            "UF",
            "N"
        ]
    )


if ufs_familias:

    df_uf_familia_novo = (

        pd.concat(
            ufs_familias,
            ignore_index=True
        )

        .groupby(
            [
                "Ano",
                "UF",
                "Familia_CBO",
                "Descricao"
            ],
            as_index=False
        )[
            "N"
        ]

        .sum()
    )

else:

    df_uf_familia_novo = pd.DataFrame(
        columns=[
            "Ano",
            "UF",
            "Familia_CBO",
            "Descricao",
            "N"
        ]
    )


# Percentuais internos.
if len(
    df_familia_novo
):

    total_novo = int(
        df_familia_novo[
            "N"
        ].sum()
    )

    df_familia_novo[
        "Percentual"
    ] = (
        df_familia_novo[
            "N"
        ]
        /
        total_novo
        *
        100
    )

else:

    total_novo = 0


if len(
    df_uf_novo
):

    df_uf_novo[
        "Percentual"
    ] = (
        df_uf_novo[
            "N"
        ]
        /
        df_uf_novo[
            "N"
        ].sum()
        *
        100
    )


# ============================================================
# 13. CARREGAR A REFERÊNCIA DA BASE FINAL V2 — SOMENTE 2022
# ============================================================

def identificar_coluna_base_v2(
    nomes,
    candidatos
):

    mapa = {
        normalizar_texto(
            nome
        ):
            nome
        for nome in nomes
    }


    for candidato in candidatos:

        chave = normalizar_texto(
            candidato
        )

        if chave in mapa:

            return mapa[
                chave
            ]


    return None


def agregar_base_v2_2022():

    if not os.path.exists(
        PASTA_BASE_V2
    ):

        raise FileNotFoundError(
            "A pasta da base V2 não foi encontrada:\n"
            f"{PASTA_BASE_V2}"
        )


    dataset = ds.dataset(
        PASTA_BASE_V2,
        format="parquet",
        partitioning="hive"
    )


    nomes = dataset.schema.names


    col_ano = identificar_coluna_base_v2(
        nomes,
        [
            "Ano"
        ]
    )


    col_uf = identificar_coluna_base_v2(
        nomes,
        [
            "UF"
        ]
    )


    col_fam = identificar_coluna_base_v2(
        nomes,
        [
            "Familia_CBO",
            "Família CBO"
        ]
    )


    col_y = identificar_coluna_base_v2(
        nomes,
        [
            "Y_doenca",
            "Y doença",
            "Y_doença"
        ]
    )


    faltantes = [

        nome
        for nome, valor in {
            "Ano":
                col_ano,

            "UF":
                col_uf,

            "Familia_CBO":
                col_fam,
        }.items()

        if valor is None
    ]


    if faltantes:

        raise RuntimeError(
            "Na base V2 não encontrei as colunas: "
            f"{faltantes}\n"
            f"Schema disponível: {nomes}"
        )


    # Descobrir o tipo de Ano para construir o filtro.
    tipo_ano = (
        dataset.schema.field(
            col_ano
        ).type
    )


    if pa.types.is_integer(
        tipo_ano
    ):

        valor_ano = ANO

    else:

        valor_ano = str(
            ANO
        )


    filtro = (
        ds.field(
            col_ano
        )
        ==
        valor_ano
    )


    colunas_ler = [
        col_ano,
        col_uf,
        col_fam,
    ]


    if col_y is not None:

        colunas_ler.append(
            col_y
        )


    scanner = dataset.scanner(
        columns=colunas_ler,
        filter=filtro,
        batch_size=250_000
    )


    cont_uf = Counter()

    cont_fam = Counter()

    cont_uf_fam = Counter()

    total = 0

    y_positivos = 0

    y_validos = 0


    for batch in scanner.to_batches():

        df = (
            batch
            .to_pandas()
        )


        # Segurança adicional.
        if col_ano in df.columns:

            ano_norm = (

                df[
                    col_ano
                ]

                .astype(
                    "string"
                )

                .str.extract(
                    r"(\d{4})",
                    expand=False
                )
            )


            df = df.loc[
                ano_norm
                ==
                str(
                    ANO
                )
            ].copy()


        if df.empty:

            continue


        uf = (

            df[
                col_uf
            ]

            .astype(
                "string"
            )

            .str.strip()

            .str.upper()
        )


        fam = extrair_familia_4(
            df[
                col_fam
            ]
        )


        total += len(
            df
        )


        for valor, qtd in (
            uf
            .value_counts(
                dropna=False
            )
            .items()
        ):

            cont_uf[
                str(
                    valor
                )
            ] += int(
                qtd
            )


        for valor, qtd in (
            fam
            .value_counts(
                dropna=False
            )
            .items()
        ):

            cont_fam[
                str(
                    valor
                )
            ] += int(
                qtd
            )


        temp = pd.DataFrame(
            {
                "UF":
                    uf,

                "Familia_CBO":
                    fam
            }
        )


        for (
            uf_val,
            fam_val
        ), qtd in (

            temp

            .groupby(
                [
                    "UF",
                    "Familia_CBO"
                ],
                dropna=False
            )

            .size()

            .items()
        ):

            cont_uf_fam[
                (
                    str(
                        uf_val
                    ),
                    str(
                        fam_val
                    )
                )
            ] += int(
                qtd
            )


        if (
            col_y is not None
            and
            col_y in df.columns
        ):

            y = pd.to_numeric(
                df[
                    col_y
                ],
                errors="coerce"
            )

            y_validos += int(
                y.notna()
                .sum()
            )

            y_positivos += int(
                (
                    y
                    ==
                    1
                )
                .sum()
            )


        del (
            df,
            temp
        )

        gc.collect()


    df_uf = pd.DataFrame(
        [
            {
                "UF":
                    uf,

                "N_V2":
                    qtd,
            }

            for uf, qtd
            in cont_uf.items()
        ]
    )


    df_fam = pd.DataFrame(
        [
            {
                "Familia_CBO":
                    fam,

                "N_V2":
                    qtd,
            }

            for fam, qtd
            in cont_fam.items()
        ]
    )


    df_uf_fam = pd.DataFrame(
        [
            {
                "UF":
                    uf,

                "Familia_CBO":
                    fam,

                "N_V2":
                    qtd,
            }

            for (
                uf,
                fam
            ), qtd
            in cont_uf_fam.items()
        ]
    )


    resumo_y = pd.DataFrame(
        [
            {
                "Ano":
                    ANO,

                "Total_V2":
                    total,

                "Y_doenca_disponivel":
                    col_y is not None,

                "Y_validos":
                    y_validos
                    if col_y is not None
                    else np.nan,

                "Y_positivos":
                    y_positivos
                    if col_y is not None
                    else np.nan,

                "Prevalencia_Y_doenca_pct":
                    (
                        y_positivos
                        /
                        y_validos
                        *
                        100
                    )
                    if (
                        col_y is not None
                        and
                        y_validos > 0
                    )
                    else np.nan,
            }
        ]
    )


    return (
        total,
        df_uf,
        df_fam,
        df_uf_fam,
        resumo_y,
    )


# ============================================================
# 14. EXECUTAR COMPARAÇÃO COM A V2
# ============================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "CARREGANDO REFERÊNCIA DE 2022 DA BASE FINAL V2"
)

print(
    "=" * 90
)


(
    total_v2,
    df_uf_v2,
    df_familia_v2,
    df_uf_familia_v2,
    df_y_v2,
) = agregar_base_v2_2022()


print(
    f"Total recuperado com 5 famílias: "
    f"{total_novo:,}"
)

print(
    f"Total da base final V2 em 2022: "
    f"{total_v2:,}"
)


# ============================================================
# 15. COMPARAÇÃO TOTAL
# ============================================================

df_comparacao_total = pd.DataFrame(
    [
        {
            "Ano":
                ANO,

            "N_recuperado_5_familias":
                total_novo,

            "N_base_V2":
                total_v2,

            "Diferenca_absoluta":
                (
                    total_novo
                    -
                    total_v2
                ),

            "Diferenca_pct_sobre_V2":
                (
                    (
                        total_novo
                        -
                        total_v2
                    )
                    /
                    total_v2
                    *
                    100
                )
                if total_v2 > 0
                else np.nan,

            "Razao_recuperado_V2":
                (
                    total_novo
                    /
                    total_v2
                )
                if total_v2 > 0
                else np.nan,
        }
    ]
)


# ============================================================
# 16. COMPARAÇÃO POR UF
# ============================================================

df_novo_uf_cmp = (

    df_uf_novo[
        [
            "UF",
            "N"
        ]
    ]

    .rename(
        columns={
            "N":
                "N_recuperado"
        }
    )
)


df_comparacao_uf = (

    df_novo_uf_cmp

    .merge(
        df_uf_v2,
        on="UF",
        how="outer"
    )

    .fillna(
        {
            "N_recuperado":
                0,

            "N_V2":
                0,
        }
    )
)


df_comparacao_uf[
    "Diferenca"
] = (

    df_comparacao_uf[
        "N_recuperado"
    ]

    -

    df_comparacao_uf[
        "N_V2"
    ]
)


df_comparacao_uf[
    "Diferenca_pct_sobre_V2"
] = np.where(

    df_comparacao_uf[
        "N_V2"
    ]
    >
    0,

    (
        df_comparacao_uf[
            "Diferenca"
        ]

        /

        df_comparacao_uf[
            "N_V2"
        ]

        *
        100
    ),

    np.nan
)


df_comparacao_uf = (

    df_comparacao_uf

    .sort_values(
        "UF"
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 17. COMPARAÇÃO POR FAMÍLIA CBO
# ============================================================

df_novo_fam_cmp = (

    df_familia_novo[
        [
            "Familia_CBO",
            "Descricao",
            "N"
        ]
    ]

    .rename(
        columns={
            "N":
                "N_recuperado"
        }
    )
)


df_comparacao_familia = (

    df_novo_fam_cmp

    .merge(
        df_familia_v2,
        on="Familia_CBO",
        how="outer"
    )
)


df_comparacao_familia[
    "Descricao"
] = (

    df_comparacao_familia[
        "Familia_CBO"
    ]

    .map(
        FAMILIAS_CBO
    )

    .fillna(
        df_comparacao_familia[
            "Descricao"
        ]
    )
)


df_comparacao_familia[
    "N_recuperado"
] = (

    df_comparacao_familia[
        "N_recuperado"
    ]
    .fillna(
        0
    )
)


df_comparacao_familia[
    "N_V2"
] = (

    df_comparacao_familia[
        "N_V2"
    ]
    .fillna(
        0
    )
)


df_comparacao_familia[
    "Diferenca"
] = (

    df_comparacao_familia[
        "N_recuperado"
    ]

    -

    df_comparacao_familia[
        "N_V2"
    ]
)


df_comparacao_familia[
    "Diferenca_pct_sobre_V2"
] = np.where(

    df_comparacao_familia[
        "N_V2"
    ]
    >
    0,

    (
        df_comparacao_familia[
            "Diferenca"
        ]

        /

        df_comparacao_familia[
            "N_V2"
        ]

        *
        100
    ),

    np.nan
)


df_comparacao_familia = (

    df_comparacao_familia

    .sort_values(
        "Familia_CBO"
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 18. COMPARAÇÃO UF x FAMÍLIA CBO
# ============================================================

df_novo_uf_fam_cmp = (

    df_uf_familia_novo[
        [
            "UF",
            "Familia_CBO",
            "Descricao",
            "N"
        ]
    ]

    .rename(
        columns={
            "N":
                "N_recuperado"
        }
    )
)


df_comparacao_uf_familia = (

    df_novo_uf_fam_cmp

    .merge(
        df_uf_familia_v2,
        on=[
            "UF",
            "Familia_CBO"
        ],
        how="outer"
    )
)


df_comparacao_uf_familia[
    "Descricao"
] = (

    df_comparacao_uf_familia[
        "Familia_CBO"
    ]

    .map(
        FAMILIAS_CBO
    )

    .fillna(
        df_comparacao_uf_familia[
            "Descricao"
        ]
    )
)


df_comparacao_uf_familia[
    "N_recuperado"
] = (

    df_comparacao_uf_familia[
        "N_recuperado"
    ]
    .fillna(
        0
    )
)


df_comparacao_uf_familia[
    "N_V2"
] = (

    df_comparacao_uf_familia[
        "N_V2"
    ]
    .fillna(
        0
    )
)


df_comparacao_uf_familia[
    "Diferenca"
] = (

    df_comparacao_uf_familia[
        "N_recuperado"
    ]

    -

    df_comparacao_uf_familia[
        "N_V2"
    ]
)


df_comparacao_uf_familia[
    "Diferenca_pct_sobre_V2"
] = np.where(

    df_comparacao_uf_familia[
        "N_V2"
    ]
    >
    0,

    (
        df_comparacao_uf_familia[
            "Diferenca"
        ]

        /

        df_comparacao_uf_familia[
            "N_V2"
        ]

        *
        100
    ),

    np.nan
)


df_comparacao_uf_familia = (

    df_comparacao_uf_familia

    .sort_values(
        [
            "UF",
            "Familia_CBO"
        ]
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 19. VALIDAR OS SEIS PARQUETS CANDIDATOS
# ============================================================

validacao = []


for grupo in GRUPOS:

    caminho = os.path.join(
        PASTA_SAIDA_PARQUETS,
        f"RAIS_PROFESSORES_{ANO}_{grupo}.parquet"
    )


    existe = os.path.exists(
        caminho
    )


    linhas = None

    colunas = None

    encontradas = set()

    familias_encontradas = set()

    municipio_estab_nulo = 0


    if existe:

        pf = pq.ParquetFile(
            caminho
        )


        linhas = int(
            pf.metadata.num_rows
        )


        colunas = len(
            pf.schema_arrow.names
        )


        for batch in pf.iter_batches(
            batch_size=250_000,
            columns=[
                "UF",
                "Familia_CBO",
                "Municipio_estabelecimento_codigo",
            ]
        ):

            df = (
                batch
                .to_pandas()
            )


            uf = (

                df[
                    "UF"
                ]

                .astype(
                    "string"
                )

                .dropna()

                .str.strip()

                .str.upper()
            )


            fam = (

                df[
                    "Familia_CBO"
                ]

                .astype(
                    "string"
                )

                .dropna()

                .str.extract(
                    r"(\d{4})",
                    expand=False
                )
            )


            encontradas.update(
                uf.tolist()
            )


            familias_encontradas.update(
                fam.dropna()
                .tolist()
            )


            municipio_estab_nulo += int(

                df[
                    "Municipio_estabelecimento_codigo"
                ]

                .isna()

                .sum()
            )


    esperadas = (
        UFS_ESPERADAS[
            grupo
        ]
    )


    familias_invalidas = (

        familias_encontradas
        -
        set(
            FAMILIAS_CBO.keys()
        )
    )


    validacao.append(
        {
            "Grupo":
                grupo,

            "Existe":
                existe,

            "Linhas":
                linhas,

            "Colunas":
                colunas,

            "UFs_encontradas":
                ", ".join(
                    sorted(
                        encontradas
                    )
                ),

            "UFs_invalidas":
                ", ".join(
                    sorted(
                        encontradas
                        -
                        esperadas
                    )
                ),

            "UFs_esperadas_ausentes":
                ", ".join(
                    sorted(
                        esperadas
                        -
                        encontradas
                    )
                ),

            "Familias_CBO_encontradas":
                ", ".join(
                    sorted(
                        familias_encontradas
                    )
                ),

            "Familias_CBO_invalidas":
                ", ".join(
                    sorted(
                        familias_invalidas
                    )
                ),

            "Municipio_estab_nulo":
                municipio_estab_nulo,
        }
    )


df_validacao = pd.DataFrame(
    validacao
)


# ============================================================
# 20. SALVAR TODOS OS RESULTADOS
# ============================================================

df_resumo.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "01_resumo_recuperacao_2022_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_familia_novo.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "02_distribuicao_familia_CBO_2022_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_uf_novo.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "03_distribuicao_UF_2022_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_uf_familia_novo.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "04_distribuicao_UF_familia_CBO_2022_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_comparacao_total.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "05_comparacao_total_com_base_V2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_comparacao_uf.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "06_comparacao_UF_com_base_V2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_comparacao_familia.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "07_comparacao_familia_CBO_com_base_V2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_comparacao_uf_familia.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "08_comparacao_UF_familia_CBO_com_base_V2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_y_v2.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "09_referencia_Y_doenca_base_V2_2022.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


df_validacao.to_csv(
    os.path.join(
        PASTA_RESULTADOS,
        "10_validacao_parquets_2022_v2.csv"
    ),
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 21. EXIBIR RESULTADOS PRINCIPAIS
# ============================================================

print(
    "\n"
    +
    "=" * 90
)

print(
    "COMPARAÇÃO TOTAL — 2022"
)

print(
    "=" * 90
)

display(
    df_comparacao_total
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "COMPARAÇÃO POR FAMÍLIA CBO"
)

print(
    "=" * 90
)

display(
    df_comparacao_familia
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "COMPARAÇÃO POR UF"
)

print(
    "=" * 90
)

display(
    df_comparacao_uf
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "REFERÊNCIA DO Y_DOENCA NA BASE V2"
)

print(
    "=" * 90
)

display(
    df_y_v2
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "VALIDAÇÃO DOS SEIS PARQUETS CANDIDATOS"
)

print(
    "=" * 90
)

display(
    df_validacao
)


# ============================================================
# 22. DIAGNÓSTICO AUTOMÁTICO
# ============================================================

diferenca_total = int(
    df_comparacao_total.loc[
        0,
        "Diferenca_absoluta"
    ]
)


pct_total = float(
    df_comparacao_total.loc[
        0,
        "Diferenca_pct_sobre_V2"
    ]
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "DIAGNÓSTICO"
)

print(
    "=" * 90
)


print(
    f"Recuperado com 5 famílias: "
    f"{total_novo:,}"
)


print(
    f"Base final V2 em 2022: "
    f"{total_v2:,}"
)


print(
    f"Diferença: "
    f"{diferenca_total:+,} "
    f"({pct_total:+.3f}%)"
)


estrutura_ok = (

    df_validacao[
        "Existe"
    ]
    .all()

    and

    df_validacao[
        "UFs_invalidas"
    ]
    .fillna(
        ""
    )
    .eq(
        ""
    )
    .all()

    and

    df_validacao[
        "Familias_CBO_invalidas"
    ]
    .fillna(
        ""
    )
    .eq(
        ""
    )
    .all()
)


if estrutura_ok:

    print(
        "\nEstrutura dos seis Parquets candidatos: OK."
    )

else:

    print(
        "\nATENÇÃO: há pendências estruturais "
        "nos Parquets candidatos."
    )


# Critério apenas indicativo.
# Não libera automaticamente a V3:
# a comparação por UF/Família precisa ser revisada.
if (
    estrutura_ok
    and
    abs(
        pct_total
    )
    <=
    0.1
):

    print(
        "\nO total recuperado ficou praticamente igual à V2."
    )

    print(
        "Mesmo assim, NÃO crie a V3 ainda."
    )

    print(
        "Primeiro revise as comparações por UF e Família CBO."
    )

else:

    print(
        "\nA população recuperada ainda difere da V2 "
        "ou há pendência estrutural."
    )

    print(
        "Use os arquivos 05 a 08 para localizar "
        "onde está a diferença antes de construir a V3."
    )


print(
    "\nParquets candidatos salvos em:"
)

print(
    PASTA_SAIDA_PARQUETS
)


print(
    "\nRelatórios salvos em:"
)

print(
    PASTA_RESULTADOS
)


print(
    "\nArquivos mais importantes para a próxima análise:"
)

print(
    "05_comparacao_total_com_base_V2.csv"
)

print(
    "06_comparacao_UF_com_base_V2.csv"
)

print(
    "07_comparacao_familia_CBO_com_base_V2.csv"
)

print(
    "08_comparacao_UF_familia_CBO_com_base_V2.csv"
)

print(
    "10_validacao_parquets_2022_v2.csv"
)
